# exp106 strict exp072 pf_z multiseed scale cache train

Reproduce exp072 `pf_z` with seed-1 strict parity, then extend the same implementation to multi-seed / scale cache candidates on the same train pseudo-tail rows.

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from strict_exp072_pf_z_multiseed_scale_cache import EXP072_TRAIN_FEATURES, OUTPUT_PREFIX, find_artifact, run_audit

paths = ExperimentPaths()
config = load_config()
print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('status:', get_nested(config, 'experiment.status'))
print('parent:', get_nested(config, 'lineage.parent'))
print('cache parent:', get_nested(config, 'lineage.cache_parent'))
print('strict_pf_z:', json.dumps(get_nested(config, 'model.strict_pf_z'), indent=2))
print('train dir:', paths.train_data_dir)
print('artifact dir:', paths.artifacts_dir)


## 2. Input preview

In [ ]:
cache_path = find_artifact(EXP072_TRAIN_FEATURES, get_nested(config, 'data.exp072_train_feature_cache_local'))
print('exp072 cache:', cache_path)
cache_header = pd.read_csv(cache_path, nrows=0).columns.tolist()
print('cache columns:', len(cache_header))
print([c for c in cache_header if c in {'pf_z', 'likpf_mean_d', 'likpf_scale_3_d', 'likpf_scale_5_d', 'likpf_scale_8_d', 'likpf_scale_12_d'}])
display(pd.read_csv(cache_path, nrows=5))

train_files = sorted(paths.train_data_dir.glob('*__horizontal_well.csv'))
print('train wells:', len(train_files))
if train_files:
    sample_hw = train_files[0]
    sample_tw = sample_hw.with_name(sample_hw.name.replace('__horizontal_well.csv', '__typewell.csv'))
    print('sample horizontal:', sample_hw.name)
    display(pd.read_csv(sample_hw).head())
    print('sample typewell:', sample_tw.name)
    display(pd.read_csv(sample_tw).head())


## 3. Run strict parity and multiseed audit

In [ ]:
summary = run_audit(config)
print(json.dumps(summary, indent=2, sort_keys=True)[:6000])


## 4. Metrics and artifacts

In [ ]:
artifact_dir = paths.artifacts_dir
metrics_path = artifact_dir / f'{OUTPUT_PREFIX}_candidate_metrics.csv'
quality_path = artifact_dir / f'{OUTPUT_PREFIX}_strict_pf_z_quality.csv'
bucket_path = artifact_dir / f'{OUTPUT_PREFIX}_bucket_metrics.csv'
summary_path = artifact_dir / f'{OUTPUT_PREFIX}_summary.json'
parity_path = artifact_dir / f'{OUTPUT_PREFIX}_parity_diff.csv.gz'

candidate_metrics = pd.read_csv(metrics_path)
display(candidate_metrics)
display(pd.read_csv(quality_path).head())
display(pd.read_csv(bucket_path).head(20))
parity = pd.read_csv(parity_path)
display(parity['abs_diff'].describe())
print('summary:', summary_path)
print('artifacts:', sorted(p.name for p in artifact_dir.glob(f'{OUTPUT_PREFIX}*')))
